<a href="https://colab.research.google.com/github/shreyaghora/ML/blob/main/ML02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install & Import all the necessary libraries,functions, models

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
    make_scorer
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

from imblearn.over_sampling import SMOTE, ADASYN

# G-Mean is often imported from imbalanced-learn or calculated manually
from imblearn.metrics import geometric_mean_score


Loading the dataset from .csv to pandas's dataframe

In [2]:
df = pd.read_csv('/content/drive/MyDrive/bank-additional-full.csv', sep=';')

 Data Cleaning


In [3]:
# Droping duration column to prevent leakage
df.drop('duration', axis=1, inplace=True) #drop method used for removing specified rows and column from a dataset

# For removing unknown values like 0, null, ?
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

 Data Transformation

In [4]:
#Encoding in binary[0 & 1] values i.e. Final value or, Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# One-Hot Encoding: Convert ordinary data into numeric form [Only 1 column is (hot)1 at a time in each row for N categories]
df = pd.get_dummies(df, drop_first=True)
#Scaling: Step of preprocessing
scaler = StandardScaler()

Data Splitting

In [5]:
# Split Features(X:Input variables used for train the model and make prediction) & Target(y:Variable the model is intend to predict)

X = df.drop('y', axis=1)
y = df['y']

#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Handling Imbalanced Data

In [6]:
#Smote(Sampling) Techniques
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

Models

In [7]:
models = {
    "Logistic Regression": LogisticRegression
     (
         max_iter=1000,
         class_weight='balanced',
         random_state=42,
         C=1.0
      ),
    "Decision Tree": DecisionTreeClassifier
     (
            max_depth=20,
            min_samples_split=40,
            min_samples_leaf=2,
            random_state=42

    ),
    "Random Forest": RandomForestClassifier
     (
        random_state=42,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
     ),
    "GBM": XGBClassifier
     (
         random_state=42,
         learning_rate=0.3,
         n_estimators=20
     ),
    "XGBoost": XGBClassifier
     (
         random_state=42,
         n_estimators=100,
         max_depth=3,
         learning_rate=0.3,
      ),
    "LightGBM": XGBClassifier
     (
         max_depth=5,
         learning_rate=0.3,
         n_estimators=100
     ),
    "CatBoost": XGBClassifier
    (
        n_estimators=500,
        max_depth=3,
        learning_rate=0.1,
    ),
    "SVM":SVC
    (
            random_state=42,
            C=0.1
      ),
    "KNN": KNeighborsClassifier
     (
         n_neighbors=15,
         weights='distance',

     ),
    "MLP": MLPClassifier
     (
         random_state=42,
         hidden_layer_sizes=(64, 32),
         activation='relu',
         solver='adam',
         max_iter=1000,
         learning_rate_init=0.001,
         alpha=0.0001,
         ),
    "BaggingClassifier": BaggingClassifier
     (
         random_state=42,
         n_estimators=200,
         max_samples=0.8,
         max_features=0.8
         ),
    "StackingClassifier": StackingClassifier
     (
        cv=10,
        estimators=[
            ('lr', LogisticRegression(random_state=42)),
            ('rf', RandomForestClassifier(random_state=42)),
            ('dt', DecisionTreeClassifier(random_state=42)),
        ],
        n_jobs=-1,
        passthrough=True,
        verbose=1,
     )

  }


For finding specificity and gmean


In [8]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return np.sqrt(sensitivity * specificity)

Performance and Composite metrics

In [9]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',

    # For using actual function present in makescorer
    'mcc': make_scorer(matthews_corrcoef),
    'kappa': make_scorer(cohen_kappa_score),
    #Not present logic in scikit-learn
    'specificity': make_scorer(specificity_score),
    'g_mean': make_scorer(g_mean_score)
}

Cross Validation Setup


In [10]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

For storing

In [11]:
results= []

Run Experiments & get the Result of using SMOTE(With Sampling) algo.

In [12]:
#Function to Evaluate Models
def evaluate_models(X, y, label):
    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=skf, scoring=scoring)

        metrics={
            "Model": name,
            "Accuracy":  f"{np.mean(scores['test_accuracy']):.4f}",
            "Precision": f"{np.mean(scores['test_precision']):.4f}",
            "Recall":    f"{np.mean(scores['test_recall']):.4f}",
            "F1 Score":  f"{np.mean(scores['test_f1']):.4f}",
            "ROC-AUC":   f"{np.mean(scores['test_roc_auc']):.4f}",
           "MCC":       f"{np.mean(scores['test_mcc']):.4f}",
            "Kappa":     f"{np.mean(scores['test_kappa']):.4f}",
            "Specificity": f"{np.mean(scores['test_specificity']):.4f}",
            "G-Mean": f"{np.mean(scores['test_g_mean']):.4f}"
        }
        #Store in list
        results.append(metrics)
        #Print the result
        print(f"\nModel: {name}")
        print(f"Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall:    {np.mean(scores['test_recall']):.4f}")
        print(f"F1 Score:  {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC:   {np.mean(scores['test_roc_auc']):.4f}")
        print(f"MCC:       {np.mean(scores['test_mcc']):.4f}")
        print(f"Kappa:     {np.mean(scores['test_kappa']):.4f}")
        print(f"Specificity: {np.mean(scores['test_specificity']):.4f}")
        print(f"G-Mean: {np.mean(scores['test_g_mean']):.4f}")
evaluate_models(X_train_sm, y_train_sm, "SMOTE")


Model: Logistic Regression
Accuracy:  0.7397
Precision: 0.8032
Recall:    0.6350
F1 Score:  0.7093
ROC-AUC:   0.8003
MCC:       0.4903
Kappa:     0.4794
Specificity: 0.8444
G-Mean: 0.7322

Model: Decision Tree
Accuracy:  0.9059
Precision: 0.9337
Recall:    0.8740
F1 Score:  0.9028
ROC-AUC:   0.9541
MCC:       0.8136
Kappa:     0.8119
Specificity: 0.9379
G-Mean: 0.9054

Model: Random Forest
Accuracy:  0.9403
Precision: 0.9487
Recall:    0.9310
F1 Score:  0.9397
ROC-AUC:   0.9820
MCC:       0.8808
Kappa:     0.8806
Specificity: 0.9496
G-Mean: 0.9402

Model: GBM
Accuracy:  0.9180
Precision: 0.9435
Recall:    0.8892
F1 Score:  0.9155
ROC-AUC:   0.9676
MCC:       0.8373
Kappa:     0.8359
Specificity: 0.9467
G-Mean: 0.9175

Model: XGBoost
Accuracy:  0.9300
Precision: 0.9661
Recall:    0.8912
F1 Score:  0.9271
ROC-AUC:   0.9707
MCC:       0.8625
Kappa:     0.8599
Specificity: 0.9687
G-Mean: 0.9291

Model: LightGBM
Accuracy:  0.9363
Precision: 0.9684
Recall:    0.9019
F1 Score:  0.9340
ROC-AU

Converting into dataframe

In [13]:
results_df = pd.DataFrame(results)

Save directly to Drive

In [14]:
from google.colab import drive
results_df.to_excel('/content/drive/MyDrive/mlresult02.xlsx', index=False)